# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook guides you through loading, exploring, and processing the FAIRˆ2 dataset using the `mlcroissant` library. We follow best practices for referencing entities via their `@id`, enabling robust and reproducible data workflows.

### Dataset Source
The dataset source is provided via the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata (as object, not dict)
print(f"Dataset Name: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}, Version: {dataset.metadata.version}")


## 2. Data Overview

Review available record sets, fields, and their `@id` identifiers. For robust referencing, we show how to access the structural components of the dataset.

In [ ]:
# List the available record sets and their @id
record_sets = list(dataset.metadata.recordSets)
print("Available record sets (referenced by @id):")
for rs in record_sets:
    print(f"@id: {rs['@id']}, name: {rs.get('name', 'No name')}")

# Now show fields (variables) for each record set
print("\nFields for each Record Set:")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}, Fields:")
    for field in rs.get('fields', []):
        print(f"  Field @id: {field['@id']}, name: {field.get('name', 'No name')}, dataType: {field.get('dataType', 'Unknown')}")
    print()

# Example: Show a preview of records from the first record set
if record_sets:
    rs_id = record_sets[0]['@id']
    print(f"Preview records from: {rs_id}")
    for idx, record in enumerate(dataset.records(record_set=rs_id)):
        print(record)
        if idx == 2:
            break

## 3. Data Extraction

Load data from the record sets into DataFrames for analysis. All reference is by `@id` fields only.

We show how to extract and preview the data for each available record set.

In [ ]:
dataframes = {}
record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSets]

# Extract each record set into a DataFrame
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded DataFrame for RecordSet @id: {record_set_id}, shape: {dataframes[record_set_id].shape}")

# Preview columns and first rows for the primary record set
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print("Columns (@id) in primary RecordSet DataFrame:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply processing steps such as filtering records by criteria, normalizing numeric fields, and grouping.

All columns and fields are referenced by their `@id` for reproducibility.

In [ ]:
# Select the main record set
record_set_id = main_record_set_id
df = dataframes[record_set_id]

# Identify candidate numeric field (@id)
# For demonstration, let's assume there is an age field with @id 'http://mlcommons.org/croissant/age'
numeric_field_id = None
for col in df.columns:
    if 'age' in col.lower():  # heuristic, can check schema for dataType == Float/Integer
        numeric_field_id = col
        break
if not numeric_field_id:
    numeric_field_id = df.select_dtypes(include=['float', 'int']).columns[0]
print(f"Selected numeric_field_id: {numeric_field_id}")

# Filtering: e.g., filter ages above a threshold
threshold = 50
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping by another field (@id); e.g., anatomical location
group_field_id = None
for col in df.columns:
    if 'anatomical' in col.lower():
        group_field_id = col
        break
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
    display(grouped_df.head())

## 5. Visualization

Visualize distributions and relationships between fields using `matplotlib` and `seaborn`.

All fields referenced by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Hist plot for numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), bins=15)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot (grouped)
if group_field_id:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion

We explored the FAIRˆ2 dataset using `mlcroissant` with entity references via their `@id`. We loaded metadata, reviewed structure, extracted record sets, and performed basic EDA and visualization.

- Key findings: Numeric variables (such as age) can be filtered and normalized by `@id`. Grouping by anatomical location highlights possible clinical variation.
- All operations referenced columns using their `@id`.

This approach provides a structured and reproducible way to interact with Croissant-conforming datasets in Python.
